# Lab 01 - Configuración Inicial y Workspace

**Objetivos:**
- Familiarizarse con notebooks de Databricks
- Crear DataFrames básicos
- Ejecutar transformaciones simples
- Trabajar con diferentes lenguajes

## Parte 1: Crear DataFrame Simple

In [ ]:
from pyspark.sql.functions import col, upper, when, current_timestamp

# Crear DataFrame de ejemplo
data = [
    (1, "Alice", "Ventas", 75000),
    (2, "Bob", "IT", 85000),
    (3, "Charlie", "Ventas", 65000),
    (4, "Diana", "Marketing", 70000),
    (5, "Eve", "IT", 90000)
]

columns = ["id", "nombre", "departamento", "salario"]

df = spark.createDataFrame(data, columns)

print(f"✅ DataFrame creado con {df.count()} registros")
display(df)

## Parte 2: Transformaciones Básicas

In [ ]:
# Transformaciones
df_transformed = df \
    .withColumn("nombre_upper", upper(col("nombre"))) \
    .withColumn("nivel_salarial",
                when(col("salario") < 70000, "Junior")
                .when(col("salario") < 80000, "Mid")
                .otherwise("Senior")) \
    .withColumn("fecha_proceso", current_timestamp())

display(df_transformed)

## Parte 3: Filtros y Agregaciones

In [ ]:
# Filtrar empleados de IT
df_it = df_transformed.filter(col("departamento") == "IT")
print(f"Empleados de IT: {df_it.count()}")
display(df_it)

In [ ]:
# =============================================================================
# AGREGACIONES: Calcular métricas agrupadas
# =============================================================================
# Objetivo: Obtener estadísticas por departamento (conteo, promedio, suma)

# Importar funciones de agregación
# Nota: sum se renombra a _sum para evitar conflicto con sum() de Python
from pyspark.sql.functions import avg, count, sum as _sum

# Paso 1: Agrupar por departamento usando groupBy()
df_summary = df.groupBy("departamento") \
    .agg(                                    # agg() aplica funciones de agregación
        count("*").alias("num_empleados"),       # Contar filas en cada grupo
        avg("salario").alias("salario_promedio"), # Calcular promedio de salarios
        _sum("salario").alias("total_salarios")   # Sumar todos los salarios del grupo
    ) \
    .orderBy(                                # Ordenar resultados
        "salario_promedio",                  # Por esta columna
        ascending=False                      # De mayor a menor (descendente)
    )

# Explicación paso a paso:
# 1. groupBy("departamento"): Agrupa registros con el mismo valor de departamento
# 2. .agg(...): Aplica funciones de agregación a cada grupo
# 3. .alias("nombre"): Renombra la columna resultante (sin esto sería "count(1)", etc.)
# 4. .orderBy(...): Ordena el resultado final

# 💡 EQUIVALENTE EN SQL:
# SELECT 
#   departamento,
#   COUNT(*) as num_empleados,
#   AVG(salario) as salario_promedio,
#   SUM(salario) as total_salarios
# FROM df
# GROUP BY departamento
# ORDER BY salario_promedio DESC

# Mostrar el resumen
display(df_summary)

# 📊 Otras funciones de agregación útiles:
# - min(), max(): Valores mínimo y máximo
# - stddev(): Desviación estándar
# - countDistinct(): Contar valores únicos
# - collect_list(): Recopilar valores en una lista

## Parte 4: Guardar Datos

In [ ]:
# =============================================================================
# ESCRITURA DE DATOS: Guardar DataFrame como tabla Delta
# =============================================================================
# Formato elegido: Managed Table (recomendado en Databricks con seguridad habilitada)

# Nombre de la tabla (se guarda en el metastore de Databricks)
table_name = "lab01_empleados"

# Escribir el DataFrame como tabla Delta
df_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

# Explicación de cada opción:
# - .write: Inicia operación de escritura
# - .format("delta"): Especifica formato Delta Lake
# - .mode("overwrite"): Modo de escritura
# - .saveAsTable(): Guarda como TABLA en lugar de archivos (evita problemas de DBFS)

# 🏆 VENTAJAS DE MANAGED TABLES:
# - No requiere permisos de DBFS (funciona con DBFS_DISABLED)
# - Databricks gestiona automáticamente la ubicación de archivos
# - Visible en Data Explorer y SQL
# - Integración automática con Unity Catalog (si está habilitado)
# - Más fácil de compartir entre notebooks y usuarios

# 📁 FORMATOS DISPONIBLES:
# - "delta": Delta Lake - RECOMENDADO (ACID, time travel, mejor performance)
# - "parquet": Parquet - Formato columnar eficiente
# - "csv": CSV - Para intercambio con sistemas externos
# - "json": JSON - Para APIs y datos semi-estructurados

# 🔄 MODOS DE ESCRITURA:
# - "overwrite": Elimina datos existentes y escribe nuevos
# - "append": Agrega datos sin eliminar existentes
# - "ignore": No escribe si ya existen datos
# - "error" (default): Falla si ya existen datos

# ⭐ VENTAJAS DE DELTA LAKE:
# - ACID transactions (escrituras atómicas y confiables)
# - Time Travel (ver versiones históricas de los datos)
# - Schema enforcement (valida que los datos cumplan el schema)
# - Mejor compresión y performance que Parquet


# 💡 NOTA SOBRE MANAGED TABLES:
# - Databricks gestiona la ubicación física de los archivos
# - No necesitas especificar rutas DBFS
# - Accesible vía SQL: SELECT * FROM lab01_empleados
# - Visible en Data Explorer (UI de Databricks)

print(f"✅ Tabla creada: {table_name}")
print(f"   Puedes consultarla con: SELECT * FROM {table_name}")

In [ ]:
# =============================================================================
# LECTURA DE DATOS: Cargar DataFrame desde tabla Delta
# =============================================================================
# Objetivo: Leer los datos que guardamos previamente

# Leer desde tabla Delta (dos formas equivalentes)
df_read = spark.table(table_name)  # Forma recomendada para managed tables
# df_read = spark.read.format("delta").table(table_name)  # Alternativa

# Explicación:
# - spark.table(): Lee una tabla del metastore (más simple y directo)
# - Funciona con managed tables, external tables, y vistas
# - No necesitas saber la ubicación física de los archivos

# 📖 OTRAS FORMAS DE LEER:
# df = spark.table("tabla_nombre")                          # Managed table (recomendado)
# df = spark.read.format("delta").table("tabla_nombre")     # Alternativa explícita
# df = spark.sql("SELECT * FROM tabla_nombre")              # Usando SQL
# df = spark.read.format("delta").load("/path")             # Desde path (requiere permisos)

# Verificar que la lectura fue exitosa
print(f"📊 Registros leídos: {df_read.count()}")

# Mostrar los datos leídos
display(df_read)

# 💡 DELTA LAKE - OPCIONES AVANZADAS:
# Time Travel - Leer versión específica:
# df_v1 = spark.read.format("delta").option("versionAsOf", 1).table(table_name)
# df_yesterday = spark.read.format("delta").option("timestampAsOf", "2026-05-21").table(table_name)


# 🔍 COMANDOS ÚTILES PARA TABLAS:
# spark.sql(f"SHOW TABLES")                      # Listar todas las tablas
# spark.sql(f"DESCRIBE EXTENDED {table_name}")  # Ver metadata de la tabla
# spark.sql(f"DESCRIBE HISTORY {table_name}")   # Ver historial de versiones

## Parte 5: SQL Queries

In [ ]:
# =============================================================================
# TEMP VIEW: Exponer DataFrame como tabla SQL
# =============================================================================
# Objetivo: Poder consultar el DataFrame usando SQL estándar

# Crear vista temporal llamada "empleados"
df_transformed.createOrReplaceTempView("empleados")

# Explicación:
# - Temp View: Vista temporal que mapea un DataFrame a una tabla SQL
# - Existe SOLO durante la sesión actual de Spark
# - Permite usar SQL para consultar el DataFrame
# - Si ya existe una vista con ese nombre, la reemplaza

# 📋 ALTERNATIVAS:
# df.createTempView("tabla")           # Falla si ya existe (más seguro)
# df.createOrReplaceTempView("tabla")  # Reemplaza si existe (más flexible)
# df.createGlobalTempView("tabla")     # Vista global (todas las sesiones)

# 🔍 DESPUÉS DE CREAR LA VIEW, PUEDES:
# - Usar SQL: spark.sql("SELECT * FROM empleados")
# - Usar magic command: %%sql en una celda
# - Unir con otras vistas/tablas en queries SQL

print("✅ Temp view 'empleados' creada")

# 💡 VENTAJA: SQL es familiar para muchos usuarios
# El mismo motor (Spark SQL) ejecuta tanto SQL como DataFrame API
# Performance es IDÉNTICA - usa el que prefieras

In [ ]:
%%sql
SELECT 
    departamento,
    nivel_salarial,
    COUNT(*) as num_empleados,
    AVG(salario) as salario_promedio
FROM empleados
GROUP BY departamento, nivel_salarial
ORDER BY departamento, salario_promedio DESC

## Resumen del Laboratorio

### Competencias Adquiridas:

**Fundamentos de PySpark:**
- Creación y manipulación de DataFrames desde datos en memoria
- Aplicación de transformaciones (withColumn, when, otherwise)
- Implementación de filtros y agregaciones complejas

**Persistencia de Datos:**
- Escritura de datos en formato Delta Lake usando managed tables
- Lectura de tablas Delta desde el metastore de Databricks

**Integración SQL:**
- Creación de vistas temporales para consultas SQL
- Ejecución de queries SQL sobre DataFrames de Spark

### Próximos Pasos:
- **Lab 02**: Optimización de clusters y gestión de costos
- **Lab 03**: Técnicas avanzadas de notebooks y parametrización
- **Lab 04**: Implementación de arquitectura Medallion (Bronze/Silver/Gold)